# modular_ml — end-to-end tutorial (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imendezguerra/modular_ml/blob/main/notebooks/tutorial_colab.ipynb)

> If you forked/renamed the repo, update the badge URL and `REPO_URL` (next cell) to `https://github.com/<you>/<repo>`.

This notebook mirrors the written [tutorial](../docs/docs/tutorial.md) and runs it top-to-bottom in Google Colab (CPU is fine). You will:

1. bootstrap `modular_ml` in the Colab VM,
2. generate the two synthetic datasets,
3. train the **sequential** (RNN) and **tabular** (MLP) models through the *same* pipeline,
4. visualise predictions, the loss curve and the PCA analysis,
5. run a **two-stage experiment**, and
6. see how to launch a **wandb sweep**.

**About wandb:** the template logs to Weights & Biases. So this notebook runs anywhere without an account, it defaults to `WANDB_MODE=disabled` (wandb calls become no-ops, exactly like the test suite). Flip the switch in the *Weights & Biases* cell to log to a real project.

## 0. Setup (Colab bootstrap)

When you open this notebook from GitHub, Colab loads **only the `.ipynb`** into a fresh VM — the rest of the repo (the `modular_ml` source and the `configs/`) is **not** there. So the first cell brings it in. It:

1. **clones the repo** (for the package source + the `configs/`),
2. **editable-installs it** (`pip install -e`), so the dependencies come straight from `pyproject.toml` — one source of truth, and Colab's preinstalled torch/numpy/etc. are skipped as already satisfied, and
3. **adds `src/` to `sys.path`** so `import modular_ml` works *immediately*.

> Why also `sys.path`? A fresh **editable** install isn't importable until the kernel restarts (its `.pth` is only read at interpreter startup), so the extra line lets *Run all* work without a restart. Keeping the example `configs/` at the repo root is intentional — this is a template you clone and edit.

**Set `REPO_URL` to your public repo** before running.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/imendezguerra/modular_ml.git"  # <-- EDIT to your public repo
REPO_DIR = REPO_URL.rstrip("/").split("/")[-1]
if REPO_DIR.endswith(".git"):
    REPO_DIR = REPO_DIR[:-4]

# Clone unless we're already running from inside the repo (e.g. locally).
if not os.path.isdir(REPO_DIR) and not os.path.isdir("src/modular_ml"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
REPO = REPO_DIR if os.path.isdir(REPO_DIR) else "."

# Install the package + its deps straight from pyproject.toml (single source of
# truth — Colab's preinstalled torch/numpy/scipy/pandas/sklearn/h5py/matplotlib
# are seen as satisfied and skipped).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO], check=True)

# A fresh editable install isn't importable until a kernel restart, so also put
# src/ on the path to import it right away.
sys.path.insert(0, os.path.abspath(os.path.join(REPO, "src")))

import modular_ml
print("modular_ml", modular_ml.__version__, "ready  |  repo at:", os.path.abspath(REPO))

## Weights & Biases

Leave `USE_REAL_WANDB = False` to run fully offline. Set it to `True` to log metrics, figures, checkpoints and the config to a real wandb project — you'll be asked for your API key, then each section appears as its own run.

> **Switching the flag?** Just re-run *this* cell after changing it — it closes any run still active from a previous cell, so the change takes effect without restarting the kernel. (Each training section also closes its own run when it finishes.)

In [ ]:
import os, wandb

USE_REAL_WANDB = False  # set True to log to a real wandb project

# Re-running this cell tears down any run left active from a previous cell. This
# matters when you flip the switch: a Runner only starts a new run when none is
# active, so a leftover *disabled* run would otherwise block a real one and your
# metrics would silently go nowhere. (No kernel restart needed.)
if wandb.run is not None:
    wandb.finish()

if USE_REAL_WANDB:
    os.environ.pop("WANDB_MODE", None)        # online
    # os.environ["MODULAR_ML_PROJECT"] = "modular-ml-tutorial"  # optional project name
    wandb.login()                              # paste your API key when prompted
    print("wandb online — each section below appears as its own run in your project")
else:
    os.environ["WANDB_MODE"] = "disabled"
    print("wandb disabled — running offline")

## 1. Generate the synthetic datasets

Two self-contained tasks share one on-disk format (`train.h5` / `test.h5` / `meta.yaml`):

* **sine** — frequency/amplitude-cued sine generation, time-major `[T, B, F]` (sequential).
* **blobs** — a ring of gaussian classes, `[B, F]` with integer labels (tabular).

In [ ]:
from modular_ml.constants import DATA_DIR
from modular_ml.data.data_gen import generate_sine, generate_blobs

generate_sine(DATA_DIR / "sine", seed=0)
generate_blobs(DATA_DIR / "blobs", seed=0)
print("datasets written to", DATA_DIR)

## 2. Train the sequential model

`config_sequential.yaml` composes the `seq_sine` dataset, the `model_rnn` architecture (`RNNBlock -> LinearBlock`), the `optim_mse` objective (MSE + a scheduled smoothness regulariser), and the `pca_rnn` analysis.

We load the config and reduce the epoch count so it runs quickly on Colab CPU — note that everything else (the schedule milestone, the regulariser ramp) is untouched. The recurrent weights start **frozen** and are **unfrozen** at the schedule milestone.

In [ ]:
from modular_ml.configs.main_configs import load_config
from modular_ml.constants import CONFIG_DIR
from modular_ml.pipeline.runner import Runner
import wandb

cfg = load_config(CONFIG_DIR / "runs" / "config_sequential.yaml")
cfg.model.epochs = 60                                   # smaller for Colab
cfg.model.architecture.schedule.milestones = [30]      # unfreeze the RNN at epoch 30

seq_runner = Runner(cfg, use_wandb=True)  # honours WANDB_MODE from the cell above
seq_runner.run_train()
seq_metrics = seq_runner.run_test()
if wandb.run is not None:
    wandb.finish()  # close this run so the next section logs to its own
print({k: round(v, 4) for k, v in seq_metrics.items() if "corrcoef" in k or "total" in k})

### Visualise: loss curve, a predicted wave, and the PCA of the RNN activity

These plots read the in-memory metric history and the saved `test_outputs.h5`. When `USE_REAL_WANDB=True`, the same metrics and a PCA figure are also in your wandb run.

In [ ]:
import matplotlib.pyplot as plt
from modular_ml.processing.dimensionality import pca_analysis
from modular_ml.tools.loaders import h5_load

out = h5_load(seq_runner.output_dir / "test_outputs.h5")
hist = seq_runner._metric_history

fig, ax = plt.subplots(1, 3, figsize=(15, 4))

ax[0].plot(hist["train/total_loss"])
ax[0].set(title="training loss", xlabel="epoch", ylabel="total loss")

pred, tgt = out["readout.prediction"], out["label.target"]
ax[1].plot(tgt[:, 0, 0], label="target")
ax[1].plot(pred[:, 0, 0], "--", label="prediction")
ax[1].set(title="test trial 0", xlabel="time step"); ax[1].legend()

_, proj = pca_analysis({"rnn.activity": out["rnn.activity"]}, ["rnn.activity"])
p = proj["rnn.activity"]
ax[2].scatter(p[:, 0], p[:, 1], s=4, alpha=0.3)
ax[2].set(title="PCA of rnn.activity", xlabel="PC1", ylabel="PC2")
plt.tight_layout(); plt.show()

## 3. Train the tabular model — the same pipeline

Only the configs differ: `FixedEncoderBlock -> LinearBlock`, a `cross_entropy` term plus a `requires_model` weight penalty, and `test/accuracy` as the metric. The `Runner`, data layer and loss machinery are identical.

In [ ]:
import wandb

cfg_tab = load_config(CONFIG_DIR / "runs" / "config_tabular.yaml")
cfg_tab.model.epochs = 40

tab_runner = Runner(cfg_tab, use_wandb=True)
tab_runner.run_train()
tab_metrics = tab_runner.run_test()
if wandb.run is not None:
    wandb.finish()  # close this run so the experiment/sweep below get their own
print({k: round(v, 4) for k, v in tab_metrics.items() if "accuracy" in k or "total" in k})

In [ ]:
out_tab = h5_load(tab_runner.output_dir / "test_outputs.h5")
_, proj = pca_analysis({"encoder.repr": out_tab["encoder.repr"]}, ["encoder.repr"])
p = proj["encoder.repr"]
labels = out_tab["label.label"]

plt.figure(figsize=(5, 4))
sc = plt.scatter(p[:, 0], p[:, 1], c=labels, cmap="tab10", s=10, alpha=0.7)
plt.title("learned encoder representation (PCA), coloured by class")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.colorbar(sc, label="class"); plt.show()

## 4. A two-stage experiment (pretrain → finetune)

An experiment chains runs and can initialise each stage from a previous stage's checkpoint, writing a provenance **manifest**. The shipped `experiment_two_stage.yaml` uses the full configs; here we build a tiny inline experiment (small epochs) so it runs fast, then inspect the manifest.

`finetune` sets `load_params_from: pretrain`, so stage 2 starts from stage 1's `model_trained.pt`.

In [ ]:
from modular_ml.constants import DATA_DIR
from modular_ml.pipeline.pipeline import run_experiment
from modular_ml.tools.loaders import yaml_save

run_yaml = DATA_DIR / "_tutorial_run.yaml"
exp_yaml = DATA_DIR / "_tutorial_exp.yaml"

yaml_save({
    "_target_": "modular_ml.configs.main_configs.Config", "seed": 0,
    "dataset": {"_target_": "modular_ml.configs.dataset_configs.DatasetConfig",
                 "name": "sine", "data_dir": str(DATA_DIR / "sine"), "task_type": "regression",
                 "in_dim": 2, "out_dim": 1, "in_key": "stim", "out_key": "target",
                 "label_key": "target"},
    "model": {"_target_": "modular_ml.configs.model_configs.ModelConfig",
               "n_inputs": 2, "n_outputs": 1, "in_key": "stim", "alpha": 0.2,
               "epochs": 20, "batch_size": 32,
               "architecture": {"_target_": "modular_ml.configs.model_configs.ArchitectureConfig",
                   "modules": {
                       "rnn": {"_target_": "modular_ml.configs.model_configs.BlockSpec",
                               "_block_target_": "modular_ml.models.blocks.RNNBlock", "id": "rnn",
                               "in_keys": ["stim"], "out_keys": ["activity"],
                               "in_dim": 2, "out_dim": 32,
                               "params": {"alpha": 0.2, "activation": "tanh"}},
                       "readout": {"_target_": "modular_ml.configs.model_configs.BlockSpec",
                               "_block_target_": "modular_ml.models.blocks.LinearBlock", "id": "readout",
                               "in_keys": ["rnn.activity"], "out_keys": ["prediction"],
                               "in_dim": 32, "out_dim": 1}}}},
    "optim": {"_target_": "modular_ml.configs.optim_configs.OptimizerConfig",
               "losses": {"mse": {"_target_": "modular_ml.configs.optim_configs.LossSpec",
                   "id": "mse", "criterion": "mse", "in_keys": ["readout.prediction"],
                   "label_key": "target"}}},
    "logger": {"_target_": "modular_ml.configs.log_configs.LoggerConfig"},
    "processing": {"_target_": "modular_ml.configs.processing_configs.ProcessingConfig",
                    "run_pca": False},
}, run_yaml)

yaml_save({
    "_target_": "modular_ml.configs.experiment_configs.ExperimentConfig",
    "runs": {
        "pretrain": {"_target_": "modular_ml.configs.experiment_configs.RunSpec",
                      "id": "pretrain", "config_path": str(run_yaml), "run_mode": "train_test"},
        "finetune": {"_target_": "modular_ml.configs.experiment_configs.RunSpec",
                      "id": "finetune", "config_path": str(run_yaml),
                      "load_params_from": "pretrain", "run_mode": "train_test"}},
}, exp_yaml)

manifest = run_experiment(exp_yaml)
for stage in manifest["stages"]:
    print(stage["run_spec_id"], "| init from:", stage["load_params_from"], "| run:", stage["wandb_run_name"])

## 5. Launch a hyperparameter sweep

Sweeps use native wandb sweeps and therefore **need a real wandb account** (`USE_REAL_WANDB = True`). The spec lives in `configs/sweeps/sweep_rnn.yaml`; parameter keys are dotted paths into the composed config and are merged before resolution.

The cell below is guarded so it only runs when wandb is enabled.

In [ ]:
if USE_REAL_WANDB:
    from modular_ml.pipeline.pipeline import run_sweep
    run_sweep(
        CONFIG_DIR / "runs" / "config_sequential.yaml",
        CONFIG_DIR / "sweeps" / "sweep_rnn.yaml",
        count=4, sweep_name="colab-rnn-grid",
    )
else:
    print("Set USE_REAL_WANDB = True (and re-run the wandb cell) to launch a sweep.")

## 6. Extend it — all in config

**Add a layer:** insert a block in `configs/runs/model/model_rnn.yaml` and point the next block's `in_keys` at its output — no Python:

```yaml
rnn2:
  _block_target_: modular_ml.models.blocks.RNNBlock
  id: rnn2
  in_keys: [rnn.activity]
  out_keys: [activity]
  in_dim: ${model.architecture.global_params.hidden}
  out_dim: ${model.architecture.global_params.hidden}
  params: { alpha: ${model.alpha}, activation: tanh }
```

**Add a loss term:** add a regulariser in `configs/runs/optim/optim_mse.yaml`:

```yaml
rank:
  _target_: modular_ml.configs.optim_configs.LossSpec
  id: rank
  criterion: effective_rank
  in_keys: [rnn.activity]
  reg_weight: 1.0e-3
```

For a brand-new layer type, subclass `Block` in `models/blocks.py`, implement `forward_block`, and reference it via `_block_target_`.

## Recap

You trained two different models through one pipeline, visualised their behaviour, ran a staged experiment, and saw how to sweep — all driven by config. See the full docs for [models and losses](../docs/docs/models-and-losses.md), [sweeps](../docs/docs/sweeps.md) and [experiments](../docs/docs/experiment-usage.md).